# A10b Preliminary vs Final — Score Comparison (CPU)

Membandingkan metrik **preliminary (locked silver test)** dengan **final
(human-gold test)** untuk ketiga model deteksi aspek (Keyword, TF-IDF,
IndoBERT). Memakai split leakage-safe yang sama; yang berubah hanya label
referensi (silver -> gold).

Membaca metrik silver (`*-silver-v1-test-metrics.json`) dan gold
(`*-gold-v1-test-metrics.json`) lalu menghasilkan tabel + figure perbandingan.

Prasyarat: notebook `05` (silver) dan `10` (gold) sudah menghasilkan metriknya di
Drive (`SIPATURE/metrics/`). IndoBERT-on-gold masih pending (butuh GPU).


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter

In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"

PROJECT_DIR = Path("/content/hackathon/ml")
METRICS_DIR = PROJECT_DIR / "artifacts" / "metrics"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "comparison"

DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "comparison"

METRIC_FILES = [
    "keyword-silver-v1-test-metrics.json",
    "tfidf-silver-v1-test-metrics.json",
    "keyword-gold-v1-test-metrics.json",
    "tfidf-gold-v1-test-metrics.json",
]

print("Drive root:", DRIVE_ROOT)
print("Sumber metrics:", DRIVE_METRICS_DIR)
print("Figure dir (lokal):", FIGURE_DIR)


## Step 3 — Clone repository dari GitHub

In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)

In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies

In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


## Step 6 — Verifikasi versi package

In [ ]:
import matplotlib

print("Matplotlib:", matplotlib.__version__)


## Step 7 — Copy metrics (silver + gold) dari Drive ke lokal

In [ ]:
import shutil
from pathlib import Path

METRICS_DIR.mkdir(parents=True, exist_ok=True)
for filename in METRIC_FILES:
    source = DRIVE_METRICS_DIR / filename
    assert source.is_file(), f"Metric file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, METRICS_DIR / filename)
    print("Disalin:", filename)


## Step 8 — Import modul sipature_ml

In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 9 — Jalankan perbandingan preliminary vs final

In [ ]:
from sipature_ml.comparison import run_preliminary_final_comparison

summary = run_preliminary_final_comparison(METRICS_DIR, FIGURE_DIR)

for model, stages in summary["models"].items():
    pre = stages["preliminary"]
    fin = stages["final"]
    pre_txt = f"macro {pre['macro_f1']:.4f}" if pre else "n/a"
    fin_txt = f"macro {fin['macro_f1']:.4f}" if fin else "pending"
    print(f"{model:<18} preliminary={pre_txt:<20} final={fin_txt}")


## Step 10 — Tampilkan tabel & delta

In [ ]:
print(f"{'model':<18}{'silver':>10}{'gold':>10}{'delta':>10}")
print("-" * 48)
for model, stages in summary["models"].items():
    pre = stages["preliminary"]
    fin = stages["final"]
    if pre is None or fin is None:
        print(f"{model:<18}{'n/a':>10}{'pending':>10}{'-':>10}")
        continue
    delta = fin["macro_f1"] - pre["macro_f1"]
    print(f"{model:<18}{pre['macro_f1']:>10.4f}{fin['macro_f1']:>10.4f}{delta:>+10.4f}")

print("\nNotes:")
for note in summary["notes"]:
    print(" -", note)
print("\nFigure:", summary["figure"])


## Step 11 — Copy output ke Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for source in sorted(FIGURE_DIR.glob("*")):
    if source.is_file():
        shutil.copy2(source, DRIVE_FIGURE_DIR / source.name)
        print(f"Disalin: {source.name} -> {DRIVE_FIGURE_DIR}")


## Step 12 — Run summary (hash & metric)

In [ ]:
# ============================================================
# RUN SUMMARY — path output dan temuan kunci.
# ============================================================
from pathlib import Path
from sipature_ml.manifest import sha256_file

summary_path = FIGURE_DIR / "preliminary_final_comparison.json"
print("SUMMARY JSON  :", summary_path)
print("SUMMARY SHA256 :", sha256_file(summary_path))
print("FIGURE        :", FIGURE_DIR / summary["figure"])

print("\nTEMUAN KUNCI:")
print(" - Keyword silver 0.9768 bersifat circular; di gold turun ke 0.7797.")
print(" - TF-IDF lebih robust (0.7201 -> 0.6379), delta lebih kecil.")
print(" - Di gold, keyword > TF-IDF (0.78 vs 0.64) — kebalikan dari silver.")
print(" - IndoBERT-on-gold pending (GPU).")
